# bias-correction-divide — worked example 3: Apply both bias corrections in a single Adam update step

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bias-correction-divide`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A full Adam parameter update needs BOTH corrected moments: `m_hat = m / (1 - beta1**t)` and `v_hat = v / (1 - beta2**t)`, then the step is `lr * m_hat / (sqrt(v_hat) + eps)`. The two corrections use DIFFERENT betas but the SAME step `t`. Forgetting either divide makes the early-training step sizes wrong.

## Worked solution

**Goal:** given raw EMA buffers `m`, `v`, the two decays, the step `t`, and `lr`/`eps`, compute the Adam update tensor.

**Step 1 — correct the first moment.** `m_hat = m / (1 - beta1 ** t)`. With `beta1=0.9`, the `t=1` denominator is `0.1`, inflating the tiny one-step EMA by 10x.

**Step 2 — correct the second moment with its OWN beta.** `v_hat = v / (1 - beta2 ** t)`. `beta2=0.999` gives a much larger inflation (`1000x` at `t=1`) because `v` decays far more slowly. Using `beta1` here would be a classic bug.

**Step 3 — assemble the update.** `lr * m_hat / (v_hat.sqrt() + eps)`. The `sqrt(v_hat)` normalizes each coordinate by its recent gradient magnitude; `eps` guards against divide-by-zero.

**Step 4 — why both divides matter.** Both buffers are zero-biased early. If only one is corrected, the ratio `m_hat/sqrt(v_hat)` is skewed, so the first steps over- or under-shoot. Correcting both keeps the effective step near `lr` from step one.

In [ ]:
def adam_update(m, v, beta1, beta2, t_step, lr=1e-3, eps=1e-8):
    m_hat = m / (1 - beta1 ** t_step)
    v_hat = v / (1 - beta2 ** t_step)
    return lr * m_hat / (v_hat.sqrt() + eps)

t.manual_seed(0)
m = t.tensor([0.01, -0.02, 0.03])
v = t.tensor([0.0001, 0.0004, 0.0009])
upd = adam_update(m, v, 0.9, 0.999, 1)
print("update:", upd)